# Configure

In [ ]:
from itertools import product

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

from openplaces.api import get_admin1, get_admin2, get_admin_ids, read_entities
from openplaces.io import read_parquet
from openplaces.path import cache_path
from openplaces.recipe import get_recipe_by_id

DATASETS = {
    'parcels': 'US-NC_parcel-nconemap-2025',
    'nsi': 'US_building-nsi-2022',
    'fema': 'US_building-fema-2023',
    'microsoft': 'US_building-microsoft-v2',
}

# Initalize

In [ ]:
admin1 = get_admin1(
    'US-NC', geom=True, recipe=get_recipe_by_id('US_admin-nhgis-2020_admin1')
)
admin2 = get_admin2(
    'US-NC', geom=True, recipe=get_recipe_by_id('US_admin-nhgis-2020_admin2')
)
admin_ids = list(admin2.index)

# Compute stats

In [ ]:
def get_stats(gdf):
    gdf_stats = (gdf.notnull() & ~gdf.isin(['', 0])).sum().rename('n_values').to_frame()
    gdf_stats['frac_values'] = gdf_stats['n_values'] / len(gdf)
    gdf_stats.index.name = 'variable'
    gdf_stats = gdf_stats.join(gdf.describe(percentiles=[0.5]).T.drop(columns='count'))
    return gdf_stats


def get_value_counts(gdf, nmax=100):

    columns_to_count = []
    for column in gdf.columns:
        if (
            gdf[column].dtype not in ['object', 'category']
            or '_id_' in column
            or column.endswith('id')
            or '_date_' in column
            or column.endswith('_date')
            or column.endswith('city')
            or column.endswith('postal_code')
            or column.endswith('source')
        ):
            continue

        mask = gdf[column].notnull() & gdf[column].ne('')
        if mask.any():
            mean_duplicates = gdf[mask][column].duplicated(keep=False).mean()

            if mean_duplicates > 0.9:
                columns_to_count += [column]

    if not columns_to_count:
        return None

    value_counts_list = []
    for column in columns_to_count:
        mask = gdf[column].notnull() & ~gdf[column].isin(['', 0])
        value_counts = gdf[mask][column].value_counts().head(nmax).to_frame()
        value_counts.index.name = 'value'
        value_counts['variable'] = column
        value_counts_list += [value_counts]
    return pd.concat(value_counts_list).reset_index().set_index(['variable', 'value'])

In [ ]:
stats_lists = {k: [] for k in DATASETS.keys()}
value_counts_lists = {k: [] for k in DATASETS.keys()}
for admin_id, (dataset_key, dataset_recipe) in product(admin_ids, DATASETS.items()):
    dataset = read_entities(admin_id, dataset_recipe)

    dataset_stats = get_stats(dataset)
    dataset_stats.insert(0, 'admin_id', admin_id)
    stats_lists[dataset_key] += [dataset_stats]

    dataset_value_counts = get_value_counts(dataset)
    if dataset_value_counts is not None:
        dataset_value_counts.insert(0, 'admin_id', admin_id)
        value_counts_lists[dataset_key] += [dataset_value_counts]

stats = {
    dataset_key: pd.concat(stats_lists[dataset_key])
    .reset_index()
    .set_index(['variable', 'admin_id'])
    for dataset_key in DATASETS.keys()
}

value_counts = {
    dataset_key: pd.concat(value_counts_lists[dataset_key])
    .reset_index()
    .set_index(['variable', 'admin_id', 'value'])
    for dataset_key in DATASETS.keys()
    if len(value_counts_lists[dataset_key])
}

# Variable availability

## By variable

In [ ]:
for dataset_key in DATASETS.keys():
    data = (
        stats[dataset_key]
        .groupby('variable')['frac_values']
        .mean()
        .mul(100)
        .loc[stats[dataset_key].index.get_level_values(0).unique()][::-1]
    )

    fig, ax = plt.subplots(figsize=(3, len(data) * 0.2))
    data.plot(kind='barh', ax=ax)
    ax.set_xlim(0, 100)
    ax.grid(axis='x', color='black', linewidth=0.2)
    ax.set_xlabel('average % non-null values (county)')
    ax.set_ylabel(None)
    ax.set_title(dataset_key)
    plt.plot()

## By county

In [ ]:
BINS = [0, 0.01, 0.03, 0.1, 0.2, 0.5, 0.8, 0.9, 0.97, 0.99, 1]

for dataset_key in DATASETS.keys():
    print(dataset_key)

    for variable in stats[dataset_key].index.get_level_values(0).unique():

        data = 1 - stats[dataset_key].loc[variable]['frac_values']
        if data.eq(0).all():
            continue

        fig, ax = plt.subplots(figsize=(9, 3))
        admin2.join(data).plot(
            'frac_values',
            ax=ax,
            scheme='user_defined',
            classification_kwds={'bins': BINS[1:], 'lowest': BINS[0]},
            legend=True,
            legend_kwds={
                'loc': 'center left',
                'bbox_to_anchor': (1, 0.5),
                'title': '% empty',
                'labels': [
                    f'{int(BINS[i]*100)} - {int(BINS[i+1]*100)}'
                    for i in range(len(BINS) - 1)
                ],
            },
            cmap='RdYlBu_r',
        )
        admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
        ax.set_title(f'{dataset_key}: {variable}')
        ax.axis('off')
        plt.show()

## Most frequent categories

In [ ]:
for dataset_key in DATASETS.keys():
    most_frequent_categories = (
        value_counts[dataset_key]
        .query('count > 0')
        .sort_values('count', ascending=False)
        .reset_index()
        .drop_duplicates(['variable', 'admin_id'])
        .set_index(['variable', 'admin_id'])
    )
    variables = most_frequent_categories.index.get_level_values('variable').unique()

    for variable in variables:
        most_frequent_by_county = most_frequent_categories.loc[variable]['value']
        if not len(most_frequent_by_county):
            continue

        fig, ax = plt.subplots(figsize=(9, 3))

        admin2.join(most_frequent_by_county).plot(
            'value',
            ax=ax,
            cmap='tab20',
            legend=True,
            legend_kwds={'loc': 'upper center', 'bbox_to_anchor': (0.5, 0), 'ncols': 2},
        )
        admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
        admin1.boundary.plot(ax=ax, color='black', linewidth=0.1)

        ax.set_title(f'{dataset_key}: most frequent `{variable}`')
        ax.axis('off')
        plt.show()

# Structure value

In [ ]:
%load_ext autoreload
%autoreload 2

## Join NSI to parcels

In [ ]:
import geopandas as gpd

nsi_on_parcels_list = []
for admin_id in admin_ids:
    print('.', end='')
    nsi = read_entities(admin_id, recipe=DATASETS['nsi'], geom=True)
    parcels = read_entities(admin_id, recipe=DATASETS['parcels'], geom=True)
    nsi_on_parcels = gpd.sjoin(
        nsi[['purpose_subgroup', 'structure_value', 'geometry']],
        parcels[['building_value', 'geometry']],
    ).rename(columns={'index_right': 'parcel_id'})
    nsi_on_parcels['admin_id'] = admin_id
    nsi_on_parcels_list += [nsi_on_parcels]
nsi_on_parcels = pd.concat(nsi_on_parcels_list)
print(f'{len(nsi_on_parcels):,d} NSI - parcel linkages.')
nsi_on_parcels.sample(10).sort_index()

## Single-family homes

In [ ]:
mask_unique_sfh = ~nsi_on_parcels.index.duplicated() & nsi_on_parcels[
    'purpose_subgroup'
].str.startswith('Single Family')

In [ ]:
median_values = (
    nsi_on_parcels[mask_unique_sfh]
    .groupby('admin_id')[['structure_value', 'building_value']]
    .median()
)

fig, ax = plt.subplots(figsize=(9, 3))
admin2.join(median_values).plot(
    'structure_value',
    ax=ax,
    legend=True,
    vmin=50000,
    vmax=400000,
    cmap='RdYlBu_r',
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.set_title(f'Median single-family `structure_value` (NSI)')
ax.axis('off')
plt.show()

fig, ax = plt.subplots(figsize=(9, 3))
admin2.join(median_values[median_values['building_value'].gt(0)]).plot(
    'building_value',
    ax=ax,
    legend=True,
    vmin=50000,
    vmax=400000,
    cmap='RdYlBu_r',
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.set_title(f'Median single-family `building_value` (parcels)')
ax.axis('off')
plt.show()

median_difference = (
    nsi_on_parcels[mask_unique_sfh]
    .groupby('admin_id')
    .apply(
        lambda x: (x['structure_value'] - x['building_value']).median(),
        include_groups=False,
    )
)

fig, ax = plt.subplots(figsize=(9, 3))
admin2.join(
    median_difference[median_values['building_value'].gt(0)].rename('median_difference')
).plot(
    'median_difference',
    ax=ax,
    legend=True,
    vmin=-150000,
    vmax=150000,
    cmap='RdYlBu_r',
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.set_title(f'Median building value difference (NSI - parcels)')
ax.axis('off')
plt.show()

In [ ]:
_nsi_on_parcels = nsi_on_parcels[mask_unique_sfh].sample(frac=0.1)
fig, ax = plt.subplots(figsize=(9, 4))
_nsi_on_parcels.plot(
    'structure_value',
    ax=ax,
    scheme='user_defined',
    classification_kwds={'bins': [5e4, 1e5, 1.5e5, 2e5, 3e5, 5e5, 1e6]},
    # scheme='quantiles',
    # k=9,
    markersize=1,
    cmap='RdYlBu_r',
    legend=True,
    legend_kwds={'loc': 'center left', 'bbox_to_anchor': (1, 0.5), 'fmt': '{0:,.0f}'},
    alpha=0.5,
    linewidth=0,
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.set_title('Single-family structure value (NSI)')
ax.axis('off')
plt.show()

In [ ]:
_nsi_on_parcels = nsi_on_parcels[mask_unique_sfh].sample(frac=0.1)
fig, ax = plt.subplots(figsize=(9, 4))
_nsi_on_parcels[_nsi_on_parcels['building_value'].gt(0)].plot(
    'building_value',
    ax=ax,
    scheme='user_defined',
    classification_kwds={'bins': [5e4, 1e5, 1.5e5, 2e5, 3e5, 5e5, 1e6]},
    markersize=1,
    cmap='RdYlBu_r',
    legend=True,
    legend_kwds={'loc': 'center left', 'bbox_to_anchor': (1, 0.5), 'fmt': '{0:,.0f}'},
    alpha=0.5,
    linewidth=0,
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.set_title('Single-family building value (parcels)')
ax.axis('off')
plt.show()

# Exploring duplicates

In [ ]:
import warnings

warnings.filterwarnings('ignore')
duplicates_by_subgroup = (
    nsi_on_parcels.groupby('purpose_subgroup').apply(
        lambda x: x.index.duplicated(keep=False).mean()
    )[::-1]
    * 100
)
warnings.filterwarnings('default')
ax = duplicates_by_subgroup.plot(
    kind='barh', figsize=(3, len(duplicates_by_subgroup) * 0.2)
)
ax.set_xlabel('% duplicate parcel linked to NSI')
ax.set_ylabel(None)

In [ ]:
warnings.filterwarnings('ignore')
duplicate_parcels_by_admin_id = (
    nsi_on_parcels.groupby('admin_id').apply(
        lambda x: x.index.duplicated(keep=False).mean()
    )[::-1]
    * 100
).rename('duplicate_parcels')
warnings.filterwarnings('default')

fig, ax = plt.subplots(figsize=(9, 4))
admin2.join(duplicate_parcels_by_admin_id).plot(
    'duplicate_parcels',
    ax=ax,
    # scheme='quantiles',
    scheme='user_defined',
    # k=10,
    classification_kwds={'bins': [0, 0.1, 1, 3, 5, 10, 25, 50]},
    cmap='RdYlBu_r',
    legend=True,
    legend_kwds={
        'loc': 'center left',
        'bbox_to_anchor': (1, 0.5),
        'title': '% duplicates',
        # 'labels': [
        # f'{int(BINS[i]*100)} - {int(BINS[i+1]*100)}'
        # for i in range(len(BINS) - 1)
        # ],
    },
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.axis('off')
ax.set_title(f'Duplicate parcels linked to NSI points')

In [ ]:
# Starting point of trying to understand the polygon duplicates

# from openplaces.api import read_entities

# parcels = read_entities('US-NC-BU', 'US-NC_parcel-nconemap-2025', geom=True)

In [ ]:
# PURPOSE_SUBGROUP = 'Single Family, 1 story, no basement'
PURPOSE_SUBGROUP = 'Multi-Family, 2 units'

mask_duplicates_in_group = nsi_on_parcels['purpose_subgroup'].eq(
    PURPOSE_SUBGROUP
) & nsi_on_parcels.index.duplicated(keep=False)
nsi_on_parcels[mask_duplicates_in_group].head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
nsi_on_parcels[mask_duplicates_in_group].plot(
    markersize=0.5, alpha=0.5, ax=ax, linewidth=0
)
admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
ax.axis('off')
ax.set_title(f'Duplicates: {PURPOSE_SUBGROUP}')

In [ ]:
XMIN = 10000
XMAX = 10000000

_nsi_on_parcels = nsi_on_parcels.sample(frac=0.1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(
    _nsi_on_parcels['structure_value'],
    _nsi_on_parcels['building_value'],
    s=0.2,
    linewidth=0,
    alpha=0.15,
)
ax.set_yscale('log')
ax.set_xscale('log')
ax.plot([XMIN, XMAX], [XMIN, XMAX], color='red', linewidth=0.5)
ax.set_xlim(XMIN, XMAX)
ax.set_ylim(XMIN, XMAX)
ax.set_ylabel('parcel: building value')
ax.set_xlabel('NSI: structure value')

In [ ]:
# NSI: "Single-family", Parcel: huge with tons of non-single-family homes
nsi_on_parcels.loc[[546034337]]

In [ ]:
from openplaces.viz.maps import show_building

# nsi_sample = nsi.sort_values('structure_value').head(1)
nsi_sample = nsi_on_parcels.loc[[546034337]]
# nsi_sample = nsi_on_parcels[mask_duplicates_in_group].sample()

admin_id = nsi_sample['admin_id'].iloc[0]
parcels = read_entities(admin_id, 'US-NC_parcel-nconemap-2025', geom=True)
# buildings_fema = read_entities(admin_id, 'US_building-fema-2023', geom=True)
buildings_nsi = read_entities(admin_id, 'US_building-nsi-2022', geom=True)
# buildings_microsoft = read_entities(admin_id, 'US_building-microsoft-v2', geom=True)

show_building(
    nsi_sample,
    geodatasets={'buildings_nsi': buildings_nsi, 'parcels': parcels},
    radius=600,
)